# KLSE Index Movement Classification Using LSTM

This notebook demonstrates building a binary classifier that predicts whether the next trading-day closing price of the Kuala Lumpur Stock Exchange (KLSE) index will move up (1) or stay the same / move down (0).

High-level parts (so you can run and review step-by-step):
- Part 1 — Setup & Imports
- Part 2 — Data download
- Part 3 — Data inspection & saving
- Part 4 — Exploratory plots
- Part 5 — Target creation
- Part 6 — Feature selection & scaling
- Part 7 — Sequence creation for LSTM
- Part 8 — Train / Test split
- Part 9 — Model building
- Part 10 — Compile & train
- Part 11 — Evaluation & visualization
- Part 12 — Save results & next steps

Notes:
- This notebook uses yfinance, pandas, scikit-learn and TensorFlow/Keras. Make sure these packages are installed in your environment.

# Part 1 — Setup & Imports

Import required libraries and set any global configuration. If you run this notebook from a different working directory, adjust paths in the save/load cells accordingly.

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

# Part 2 — Download KLSE data from Yahoo Finance

Define the KLSE ticker symbol and the date range to download. Modify the ticker or date range if you want data for other assets or periods.

In [ ]:
# Define the KLSE ticker
ticker = "^KLSE"

In [ ]:
# Download historical data for the KLSE index
df = yf.download(
    ticker,
    start="2005-01-01",
    end="2026-01-01",
    interval="1d"
)

# Part 3 — Data Inspection & Saving

Inspect the downloaded DataFrame to confirm successful retrieval and to get an initial sense of the data (rows, columns, missing values, basic statistics). Save a clean copy for reproducibility.

In [ ]:
# Display first 5 rows
print(df.head())

In [ ]:
# Display last 5 rows
print(df.tail())

In [ ]:
# Check dataset size
print("Dataset shape:", df.shape)

In [ ]:
# Check column names
print("Columns:")
print(df.columns)

In [ ]:
# Check dataset information
print("\nDataset information:")
print(df.info())

In [ ]:
# Check missing values
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Show basic statistical summary
print("\nStatistical summary:")
print(df.describe())

In [ ]:
# Save the dataset to a CSV file
# move up to project root if notebook is in Notebook/
%cd ..
# now ./Dataset refers to project_root/Dataset
df.to_csv("./Dataset/KLSE_20_years_data.csv", index=False)

print("Dataset saved successfully as KLSE_20_years_data.csv")

# Part 4 — Exploratory Plots

Plot the Close price over time to visualize the long-term movement of the KLSE index. Use additional plots here for further EDA if desired.

In [ ]:
# Plot the closing price over time
plt.figure(figsize=(12, 6))
plt.plot(df.index, df['Close'], label='KLSE Closing Price')
plt.title('KLSE Closing Price Over Time')
plt.xlabel('Date')
plt.ylabel('Closing Price (MYR)')
plt.legend()
plt.grid()
plt.show()

# Part 5 — Target Creation

Create the binary target variable. The target is 1 when the next trading day's Close is greater than the current day's Close, otherwise 0. Drop the last row which has no next-day label.

In [ ]:
# Create target variable
# 1 = Next trading day closing price is higher than current day closing price
# 0 = Next trading day closing price is lower than or equal to current day closing price

df["Target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

# Remove the last row because there is no next trading day closing price
df = df.dropna()

# Check target result
print(df[["Close", "Target"]].head())
print(df[["Close", "Target"]].tail())

# Check class distribution
print(df["Target"].value_counts())

# Part 6 — Feature Selection & Scaling

Select input features for the model. Here we use common OHLCV features: Open, High, Low, Close and Volume. Then normalize the features to [0,1] with MinMaxScaler.

In [ ]:
# Select input features for LSTM model
features = ["Open", "High", "Low", "Close", "Volume"]

X = df[features]
y = df["Target"]

# Display selected features
print(X.head())
print(y.head())

# Check feature shape
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Normalise the input features into range 0 to 1
scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

# Check scaled data
print(X_scaled[:5])
print("Scaled feature shape:", X_scaled.shape)

# Part 7 — Sequence Creation for LSTM

Create time-series sequences for LSTM input. We use a lookback window of 60 trading days (time_steps = 60). The function below converts the scaled feature matrix and labels into (samples, time_steps, features) and corresponding labels.

In [ ]:
import numpy as np

# Function to create LSTM sequences
def create_sequences(X, y, time_steps=60):
    Xs, ys = [], []

    for i in range(time_steps, len(X)):
        Xs.append(X[i-time_steps:i])
        ys.append(y.iloc[i])

    return np.array(Xs), np.array(ys)


# Use previous 60 trading days to predict next trading day movement
time_steps = 60

X_seq, y_seq = create_sequences(X_scaled, y, time_steps)

print("X sequence shape:", X_seq.shape)
print("y sequence shape:", y_seq.shape)

# Part 8 — Train / Test Split

Split sequences into training and testing sets. We use an 80/20 split (no random shuffle to preserve time order). This is important for time-series forecasting/classification to avoid look-ahead bias.

In [ ]:
# Split data into training and testing sets
# 80% training data, 20% testing data

train_size = int(len(X_seq) * 0.8)

X_train = X_seq[:train_size]
X_test = X_seq[train_size:]

y_train = y_seq[:train_size]
y_test = y_seq[train_size:]

# Check data shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

# Part 9 — Model Building

Build an LSTM-based Keras sequential model for binary classification. The architecture below is a simple baseline: two LSTM layers with dropout and a sigmoid output neuron.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Build LSTM model
model = Sequential()

# First LSTM layer
model.add(LSTM(
    units=64,
    return_sequences=True,
    input_shape=(X_train.shape[1], X_train.shape[2])
))
model.add(Dropout(0.2))

# Second LSTM layer
model.add(LSTM(
    units=32,
    return_sequences=False
))
model.add(Dropout(0.2))

# Output layer for binary classification
model.add(Dense(1, activation="sigmoid"))

# Display model architecture
model.summary()

# Part 10 — Compile & Train

Compile the model with Adam optimizer and binary cross-entropy loss. Accuracy is used as a simple performance metric; consider adding AUC for imbalanced datasets. Then train the model.

In [ ]:
# Compile the LSTM model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")

In [ ]:
# Train the LSTM model
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Part 11 — Evaluation & Visualization

Plot training/validation accuracy and loss to inspect training behavior for signs of overfitting or underfitting. After training, make predictions on the test set and calculate evaluation metrics.

In [ ]:
import matplotlib.pyplot as plt

# Plot training and validation accuracy
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("LSTM Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Plot training and validation loss
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("LSTM Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

# Make predictions on the test set and convert predicted probabilities to class labels using a 0.5 threshold. You can change the threshold to tune precision/recall trade-offs.

In [ ]:
# Make predictions on the test dataset
y_pred_prob = model.predict(X_test)

# Convert prediction probabilities into class labels
# If probability is greater than 0.5, classify as 1
# Otherwise, classify as 0
y_pred = (y_pred_prob > 0.5).astype(int)

# Display first 10 prediction probabilities
print("First 10 prediction probabilities:")
print(y_pred_prob[:10])

# Display first 10 predicted classes
print("\nFirst 10 predicted classes:")
print(y_pred[:10])

# Display first 10 actual classes
print("\nFirst 10 actual classes:")
print(y_test[:10])

# Prepare predictions and evaluate model performance using accuracy, precision, recall and F1. The notebook uses zero_division=0 in metrics to avoid exceptions for empty predicted classes.

In [ ]:
# Flatten predicted labels for evaluation
y_pred = y_pred.flatten()

# Check shapes
print("y_test shape:", y_test.shape)
print("y_pred shape:", y_pred.shape)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

# Print results
print("Model Evaluation Results")
print("------------------------")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)

# Visualize the confusion matrix and compare actual vs predicted movements for a subset of test samples.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Plot confusion matrix
plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Down/Unchanged", "Up"],
    yticklabels=["Down/Unchanged", "Up"]
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.show()

In [ ]:
# Plot actual vs predicted movement for first 100 test samples
plt.figure(figsize=(12, 5))
plt.plot(y_test[:100], label="Actual Movement", marker="o")
plt.plot(y_pred[:100], label="Predicted Movement", marker="x")

plt.title("Actual vs Predicted KLSE Movement")
plt.xlabel("Test Sample")
plt.ylabel("Movement Class")
plt.yticks([0, 1], ["Down/Unchanged", "Up"])
plt.legend()
plt.grid(True)
plt.show()

# Part 12 — Save Results & Next Steps

Save the trained model and evaluation results (evaluation metrics, prediction results, confusion matrix) to the project's Result folder so outputs can be reviewed and reused. Also includes suggestions for next steps.

In [ ]:
# Save the trained LSTM model
model.save("./Result/KLSE_LSTM_model.h5")

print("Model saved successfully as KLSE_LSTM_model.h5")

In [ ]:
# Save evaluation metrics into a dictionary
evaluation_results = {
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1
}

# Convert to DataFrame
evaluation_df = pd.DataFrame([evaluation_results])

# Save as CSV
evaluation_df.to_csv("./Result/KLSE_LSTM_evaluation_results.csv", index=False)

print("Evaluation results saved successfully.")

In [ ]:
# Save actual and predicted results
prediction_results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

# Save as CSV
prediction_results.to_csv("./Result/KLSE_LSTM_prediction_results.csv", index=False)

print("Prediction results saved successfully.")

In [ ]:
# Save confusion matrix as CSV
confusion_matrix_df = pd.DataFrame(
    cm,
    index=["Actual Down/Unchanged", "Actual Up"],
    columns=["Predicted Down/Unchanged", "Predicted Up"]
)

confusion_matrix_df.to_csv("./Result/KLSE_LSTM_confusion_matrix.csv")

print("Confusion matrix saved successfully.")

# Next steps suggestions:
# - Add EarlyStopping and ModelCheckpoint callbacks to prevent overfitting and persist the best model
# - Consider class weighting or resampling if the class imbalance significantly impacts recall/precision
# - Experiment with different lookback windows, feature engineering, or alternative models (CNN, Transformer)
# - Add a requirements.txt and a short README describing how to run the notebook
